In [3]:
import numpy as np
from sklearn.model_selection import cross_val_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.datasets import load_iris

# Dataset
data = load_iris()
X, y = data.data, data.target
print(f"X shape: {X.shape}")
print(X[:5])

# Parameters
population_size = 20
num_features = X.shape[1]
generations = 50
mutation_rate = 0.1

# Fitness function
def fitness_function(chromosome):
    selected_features = np.where(chromosome == 1)[0]
    if len(selected_features) == 0:
        return 0  # Penalize empty subsets
    X_selected = X[:, selected_features]
    model = DecisionTreeClassifier()
    scores = cross_val_score(model, X_selected, y, cv=5)
    return scores.mean()

# Initialize population
population = np.random.randint(2, size=(population_size, num_features))

# Evolution loop
best_fitness = 0
for generation in range(generations):
    # Evaluate fitness
    fitness_scores = np.array([fitness_function(chromosome) for chromosome in population])
    # Selection
    parents = population[np.argsort(fitness_scores)[-2:]]  # Top 2 parents
    # Crossover
    offspring = []
    for _ in range(population_size // 2):
        p1, p2 = parents[np.random.randint(2)], parents[np.random.randint(2)]
        cross_point = np.random.randint(1, num_features - 1)
        child1 = np.concatenate([p1[:cross_point], p2[cross_point:]])
        child2 = np.concatenate([p2[:cross_point], p1[cross_point:]])
        offspring.extend([child1, child2])
    # Mutation
    offspring = np.array(offspring)
    for individual in offspring:
        if np.random.rand() < mutation_rate:
            mutation_point = np.random.randint(num_features)
            individual[mutation_point] = 1 - individual[mutation_point]
    # New generation
    population = offspring

    # Update best fitness
    best_fitness = max(best_fitness, np.max(fitness_scores))


# Best solution
best_chromosome = population[np.argmax(fitness_scores)]
selected_features = np.where(best_chromosome == 1)[0]
print("Selected features:", selected_features)
print("Best fitness:", best_fitness)

X shape: (150, 4)
[[5.1 3.5 1.4 0.2]
 [4.9 3.  1.4 0.2]
 [4.7 3.2 1.3 0.2]
 [4.6 3.1 1.5 0.2]
 [5.  3.6 1.4 0.2]]
Selected features: [1 2 3]
Best fitness: 0.9666666666666668
